# NLP Assignment 2

**Student Name:** Abubakar Imran  
**Roll Number:** i23-2589

---

In [34]:
# Initial Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import collections
import os
import json
import math
import random

## 3. Sequence Modeling: BiLSTM + CRF

In [35]:
def read_conll(file_path):
    sentences = []
    current_sentence = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.strip().split("\t")
                if len(parts) == 2:
                    current_sentence.append((parts[0], parts[1]))
    return sentences

pos_train = read_conll("data/pos_train.conll")
ner_train = read_conll("data/ner_train.conll")

print(f"Loaded {len(pos_train)} POS sentences.")
print(f"Loaded {len(ner_train)} NER sentences.")

Loaded 400 POS sentences.
Loaded 400 NER sentences.


In [36]:
from models.bilstm import POSModel, NERModel

def create_tag_map(sentences):
    tags = set()
    for sent in sentences:
        for word, tag in sent:
            tags.add(tag)
    sorted_tags = sorted(list(tags))
    tag2idx = {tag: i for i, tag in enumerate(sorted_tags)}
    idx2tag = {i: tag for tag, i in tag2idx.items()}
    return tag2idx, idx2tag

pos_tag2idx, pos_idx2tag = create_tag_map(pos_train)
ner_tag2idx, ner_idx2tag = create_tag_map(ner_train)

print(f"POS Tags: {len(pos_tag2idx)}")
print(f"NER Tags: {len(ner_tag2idx)}")

POS Tags: 9
NER Tags: 4


In [37]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix.get(w, to_ix.get("<UNK>", 0)) for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

# 1. POS TAGGING TRAINING
EMBEDDING_DIM = 100
HIDDEN_DIM = 128

pos_model = POSModel(len(word2idx), EMBEDDING_DIM, HIDDEN_DIM, len(pos_tag2idx))
w2v_weight = torch.from_numpy(np.load("embeddings/embeddings_w2v.npy"))
pos_model.encoder.embedding.weight.data.copy_(w2v_weight)

loss_function = nn.NLLLoss()
optimizer = torch.optim.Adam(pos_model.parameters(), lr=0.01)

print("Training POS Model...")
for epoch in range(5):
    total_loss = 0
    for sentence in pos_train:
        words = [w[0] for w in sentence]
        tags = [w[1] for w in sentence]
        pos_model.zero_grad()
        sentence_in = prepare_sequence(words, word2idx).unsqueeze(0)
        targets = prepare_sequence(tags, pos_tag2idx)
        tag_scores = pos_model(sentence_in).squeeze(0)
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(pos_train):.4f}")

torch.save(pos_model.state_dict(), "models/bilstm_pos.pt")
print("POS Model saved.")

Training POS Model...
Epoch 1, Loss: 0.3132
Epoch 2, Loss: 0.0293
Epoch 3, Loss: 0.0247
Epoch 4, Loss: 0.0085
Epoch 5, Loss: 0.0072
POS Model saved.


In [38]:
# 2. NER TRAINING (With Manual CRF)
ner_model = NERModel(len(word2idx), EMBEDDING_DIM, HIDDEN_DIM, len(ner_tag2idx))
ner_model.encoder.embedding.weight.data.copy_(w2v_weight)

optimizer = torch.optim.Adam(ner_model.parameters(), lr=0.01)

print("Training NER Model with CRF...")
for epoch in range(5):
    total_loss = 0
    for sentence in ner_train:
        words = [w[0] for w in sentence]
        tags = [w[1] for w in sentence]
        ner_model.zero_grad()
        sentence_in = prepare_sequence(words, word2idx).unsqueeze(0)
        targets = prepare_sequence(tags, ner_tag2idx)
        loss = ner_model.neg_log_likelihood(sentence_in, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(ner_train):.4f}")

torch.save(ner_model.state_dict(), "models/bilstm_ner.pt")
print("NER Model saved.")

Training NER Model with CRF...
Epoch 1, Loss: 2.2705
Epoch 2, Loss: 0.5828
Epoch 3, Loss: 0.4082
Epoch 4, Loss: 0.3521
Epoch 5, Loss: 0.2184
NER Model saved.
